# NCA Workbench

Notebook для локального и серверного запуска baseline NCA.

- Данные на диске: `[T, H, W, F_data]`
- Состояние модели: `[B, data_channels + hidden_channels, H, W]`
- `primary_channel` используется для публичных метрик и визуализаций
- `loss_channels` управляет supervised loss: `primary` или `all_observed`
- Все импорты идут только через пакет `NCA`
- Все пути строятся только через `PROJECT_ROOT`


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "NCA" / "__init__.py").exists():
            return candidate
        if (candidate / "Real_game_of_life" / "NCA" / "__init__.py").exists():
            return candidate / "Real_game_of_life"
        if (candidate / "NCA" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing NCA/__init__.py")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("cwd =", Path.cwd().resolve())
print("project_root =", PROJECT_ROOT)
print("sys.path[0:3] =", sys.path[:3])


cwd = D:\Proga\Game_of_life\Real_game_of_life\NCA
project_root = D:\Proga\Game_of_life\Real_game_of_life
sys.path[0:3] = ['D:\\Proga\\Game_of_life\\Real_game_of_life', 'd:\\Proga\\Game_of_life\\Real_game_of_life\\NCA', 'd:\\Anaconda3\\NewAnaconda\\python312.zip']


In [2]:
import numpy as np
import torch

from NCA.dataset import build_dataloaders, discover_npy_files
from NCA.model import NCA
from NCA.train import deterministic_eval, stochastic_eval, train_epoch
from NCA.utils import (
    build_initial_state,
    get_device,
    load_checkpoint,
    rollout_model,
    save_checkpoint,
    set_seed,
)
from NCA.visualize import (
    plot_metric_curves,
    plot_triptych,
    plot_uncertainty_heatmap,
    save_rollout_animation,
)


In [3]:
CONFIG = {
    "project_root": PROJECT_ROOT,
    "data_root": PROJECT_ROOT / "NCA" / "data",
    "pattern": "*.npy",
    "run_dir": PROJECT_ROOT / "NCA" / "runs" / "workbench",
    "split_mode": "within_file",
    "split_ratios": (0.8, 0.1, 0.1),
    "train_steps": (2, 2),
    "eval_steps": {
        "one_step": 1,
        "rollout": 2,
        "stochastic": 2,
    },
    "batch_size": 8,
    "epochs": 5,
    "lr": 1e-3,
    "kernel_size": 3,
    "model_width": 64,
    "update_prob": 0.5,
    "data_channels": 1,
    "hidden_channels": 1,
    "primary_channel": 0,
    "loss_channels": "primary",
    "num_rollouts": 8,
    "loss_mode": "hybrid",
    "lambda_intermediate": 0.5,
    "lambda_hidden_l2": 1e-4,
    "use_alive_mask": False,
    "seed": 0,
}

CONFIG["data_root"].mkdir(parents=True, exist_ok=True)
CONFIG["run_dir"].mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
device = get_device()

print("device =", device)
print("data_root =", CONFIG["data_root"])
print("run_dir =", CONFIG["run_dir"])


device = cpu
data_root = D:\Proga\Game_of_life\Real_game_of_life\NCA\data
run_dir = D:\Proga\Game_of_life\Real_game_of_life\NCA\runs\workbench


In [4]:
out_dir = CONFIG["data_root"]
out_dir.mkdir(parents=True, exist_ok=True)

T, H, W = 32, 64, 64
data = np.zeros((T, H, W, 1), dtype=np.float32)

cy, cx = H // 2, W // 2
data[0, cy, cx, 0] = 1.0
data[0, cy - 1, cx, 0] = 0.7
data[0, cy + 1, cx, 0] = 0.7
data[0, cy, cx - 1, 0] = 0.7
data[0, cy, cx + 1, 0] = 0.7

for t in range(1, T):
    prev = data[t - 1, ..., 0]
    neighbors = (
        np.roll(prev, 1, axis=0)
        + np.roll(prev, -1, axis=0)
        + np.roll(prev, 1, axis=1)
        + np.roll(prev, -1, axis=1)
        + np.roll(np.roll(prev, 1, axis=0), 1, axis=1)
        + np.roll(np.roll(prev, 1, axis=0), -1, axis=1)
        + np.roll(np.roll(prev, -1, axis=0), 1, axis=1)
        + np.roll(np.roll(prev, -1, axis=0), -1, axis=1)
    ) / 8.0

    growth = 0.18 * neighbors * (1.0 - prev)
    decay = 0.06 * prev
    smooth = 0.12 * (neighbors - prev)
    data[t, ..., 0] = np.clip(prev + growth - decay + smooth, 0.0, 1.0)

out_path = out_dir / "synthetic_colony.npy"
np.save(out_path, data)

print("saved:", out_path)
print("shape:", data.shape)
print("dtype:", data.dtype)
print("min/max:", float(data.min()), float(data.max()))


saved: D:\Proga\Game_of_life\Real_game_of_life\NCA\data\synthetic_colony.npy
shape: (32, 64, 64, 1)
dtype: float32
min/max: 0.0 1.0


In [5]:
files = discover_npy_files(CONFIG["data_root"], CONFIG["pattern"])
print(f"Found {len(files)} files")
sample = np.load(files[0])
print("Example shape:", sample.shape)
print("Expected disk format [T, H, W, F_data]")
print("Configured data_channels:", CONFIG["data_channels"])


Found 1 files
Example shape: (32, 64, 64, 1)
Expected disk format [T, H, W, F_data], with F_data=1 in v1


In [6]:
loaders, normalizer = build_dataloaders(
    data_root=CONFIG["data_root"],
    pattern=CONFIG["pattern"],
    split_mode=CONFIG["split_mode"],
    split_ratios=CONFIG["split_ratios"],
    train_steps=CONFIG["train_steps"],
    eval_steps=CONFIG["eval_steps"],
    batch_size=CONFIG["batch_size"],
    eval_batch_size=CONFIG["batch_size"],
    seed=CONFIG["seed"],
    data_channels=CONFIG["data_channels"],
    primary_channel=CONFIG["primary_channel"],
)

train_batch = next(iter(loaders["train"]))
print("input_visible:", train_batch["input_visible"].shape)
print("targets_visible:", train_batch["targets_visible"].shape)
print("horizons:", train_batch["horizons"])


input_visible: torch.Size([8, 1, 64, 64])
targets_visible: torch.Size([8, 2, 1, 64, 64])
horizons: tensor([2, 2, 2, 2, 2, 2, 2, 2])


In [7]:
model = NCA(
    state_channels=CONFIG["data_channels"] + CONFIG["hidden_channels"],
    model_width=CONFIG["model_width"],
    kernel_size=CONFIG["kernel_size"],
    update_prob=CONFIG["update_prob"],
    use_alive_mask=CONFIG["use_alive_mask"],
    primary_channel=CONFIG["primary_channel"],
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
model


NCA(
  (conv1): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)

In [ ]:
import base64

from IPython.display import HTML, Markdown, clear_output, display
from tqdm.auto import tqdm

history = []
epoch_bar = tqdm(range(CONFIG["epochs"]), desc="epochs")

for epoch in epoch_bar:
    train_metrics = train_epoch(
        model,
        loaders["train"],
        optimizer,
        device=device,
        loss_mode=CONFIG["loss_mode"],
        lambda_intermediate=CONFIG["lambda_intermediate"],
        lambda_hidden_l2=CONFIG["lambda_hidden_l2"],
        data_channels=CONFIG["data_channels"],
        hidden_channels=CONFIG["hidden_channels"],
        loss_channels=CONFIG["loss_channels"],
        primary_channel=CONFIG["primary_channel"],
        show_progress=True,
        progress_desc=f"train epoch {epoch}",
    )

    val_det = deterministic_eval(
        model,
        loaders["val_rollout"],
        device=device,
        data_channels=CONFIG["data_channels"],
        hidden_channels=CONFIG["hidden_channels"],
        primary_channel=CONFIG["primary_channel"],
        show_progress=True,
        progress_desc=f"det eval epoch {epoch}",
    )

    val_stoch = stochastic_eval(
        model,
        loaders["val_rollout"],
        device=device,
        num_rollouts=CONFIG["num_rollouts"],
        data_channels=CONFIG["data_channels"],
        hidden_channels=CONFIG["hidden_channels"],
        primary_channel=CONFIG["primary_channel"],
        show_progress=True,
        progress_desc=f"stoch eval epoch {epoch}",
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_det_{k}": v for k, v in val_det.items()},
        **{f"val_stoch_{k}": v for k, v in val_stoch.items()},
    }
    history.append(row)

    epoch_bar.set_postfix(
        train_loss=f"{row['train_loss']:.3e}",
        det_roll=f"{row['val_det_rollout_mse']:.3e}",
        stoch_exp=f"{row['val_stoch_expected_mse']:.3e}",
    )

    clear_output(wait=True)
    display(Markdown(f"## Epoch {epoch + 1}/{CONFIG['epochs']}"))
    display(row)
    display(history[-5:])

save_checkpoint(
    CONFIG["run_dir"] / "checkpoint_latest.pt",
    model,
    optimizer,
    CONFIG["epochs"] - 1,
    {"config": CONFIG, "normalizer": normalizer.state_dict()},
    history[-1],
)

history[-1]


## Epoch 5/5

{'epoch': 4,
 'train_loss': 6.097364530432969e-05,
 'train_final_loss': 5.2087755345079735e-05,
 'train_intermediate_loss': 1.6749582452272687e-05,
 'train_hidden_penalty': 0.005110987772544225,
 'val_det_one_step_mse': 0.00011534739314811304,
 'val_det_rollout_mse': 0.000282745691947639,
 'val_det_population_mass_error': 2.153970956802368,
 'val_stoch_expected_mse': 0.0001240577839780599,
 'val_stoch_ensemble_mean_mse': 9.066795610124245e-05,
 'val_stoch_pixelwise_std_mean': 0.005573278293013573,
 'val_stoch_mass_std': 0.3696203827857971,
 'val_stoch_population_mass_error': 1.1314634084701538}

[{'epoch': 0,
  'train_loss': 0.0005727087145714904,
  'train_final_loss': 0.0004904776327142221,
  'train_intermediate_loss': 0.00016442827472928911,
  'train_hidden_penalty': 0.00016927195247262716,
  'val_det_one_step_mse': 0.00020311630214564502,
  'val_det_rollout_mse': 0.0005081620765849948,
  'val_det_population_mass_error': 2.883763313293457,
  'val_stoch_expected_mse': 0.0002160950971301645,
  'val_stoch_ensemble_mean_mse': 0.00015590523253194988,
  'val_stoch_pixelwise_std_mean': 0.007494164630770683,
  'val_stoch_mass_std': 0.5325030088424683,
  'val_stoch_population_mass_error': 1.5020554065704346},
 {'epoch': 1,
  'train_loss': 0.0006523360110198458,
  'train_final_loss': 0.0005593461586007228,
  'train_intermediate_loss': 0.00018534515159747875,
  'train_hidden_penalty': 0.003172583800430099,
  'val_det_one_step_mse': 0.00013132643653079867,
  'val_det_rollout_mse': 0.0003029173531103879,
  'val_det_population_mass_error': 2.225325584411621,
  'val_stoch_expected_mse': 0.

{'epoch': 4,
 'train_loss': 6.097364530432969e-05,
 'train_final_loss': 5.2087755345079735e-05,
 'train_intermediate_loss': 1.6749582452272687e-05,
 'train_hidden_penalty': 0.005110987772544225,
 'val_det_one_step_mse': 0.00011534739314811304,
 'val_det_rollout_mse': 0.000282745691947639,
 'val_det_population_mass_error': 2.153970956802368,
 'val_stoch_expected_mse': 0.0001240577839780599,
 'val_stoch_ensemble_mean_mse': 9.066795610124245e-05,
 'val_stoch_pixelwise_std_mean': 0.005573278293013573,
 'val_stoch_mass_std': 0.3696203827857971,
 'val_stoch_population_mass_error': 1.1314634084701538}

: 

In [ ]:
def _data_uri(path: Path) -> str:
    suffix = path.suffix.lower()
    mime = {
        ".png": "image/png",
        ".gif": "image/gif",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
    }.get(suffix, "application/octet-stream")
    payload = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{payload}"


loss_curve_path = plot_metric_curves(
    history,
    CONFIG["run_dir"] / "loss_curve.png",
    ["train_loss", "val_det_rollout_mse", "val_stoch_expected_mse"],
)
display(HTML(
    f"""
    <div style='margin: 12px 0 8px;'>
      <div style='font-size: 20px; font-weight: 700; margin-bottom: 10px;'>Training Curves</div>
      <img src='{_data_uri(loss_curve_path)}' style='max-width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
    </div>
    """
))
loss_curve_path


In [ ]:
batch = next(iter(loaders["val_rollout"]))
visible = batch["input_visible"].to(device)
state0 = build_initial_state(
    visible,
    hidden_channels=CONFIG["hidden_channels"],
    hidden_init="zeros",
)

steps = int(batch["horizons"].max().item())
det_rollout = rollout_model(model, state0, steps=steps, stochastic=False)
stoch_rollouts = torch.stack(
    [
        rollout_model(model, state0, steps=steps, stochastic=True)
        for _ in range(CONFIG["num_rollouts"])
    ],
    dim=0,
)

triptych_path = plot_triptych(
    batch["input_visible"][0],
    batch["targets_visible"][0, -1],
    det_rollout[-1, 0, :CONFIG["data_channels"]].detach().cpu(),
    CONFIG["run_dir"] / "triptych.png",
    title=f"Primary observed channel #{CONFIG['primary_channel']}",
    channel_index=CONFIG["primary_channel"],
)

uncertainty_path = plot_uncertainty_heatmap(
    stoch_rollouts.detach().cpu(),
    CONFIG["run_dir"] / "uncertainty.png",
    visible_channel=CONFIG["primary_channel"],
    step_index=-1,
)

animation_path = save_rollout_animation(
    det_rollout[:, 0, :CONFIG["data_channels"]].detach().cpu(),
    CONFIG["run_dir"] / "det_rollout.gif",
    channel_index=CONFIG["primary_channel"],
)

display(HTML(
    f"""
    <div style='margin: 12px 0 8px;'>
      <div style='font-size: 20px; font-weight: 700; margin-bottom: 10px;'>Visual Diagnostics</div>
      <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(320px, 1fr)); gap: 16px;'>
        <div>
          <div style='font-size: 16px; font-weight: 600; margin-bottom: 6px;'>Triptych</div>
          <img src='{_data_uri(triptych_path)}' style='width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
        </div>
        <div>
          <div style='font-size: 16px; font-weight: 600; margin-bottom: 6px;'>Uncertainty</div>
          <img src='{_data_uri(uncertainty_path)}' style='width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
        </div>
        <div>
          <div style='font-size: 16px; font-weight: 600; margin-bottom: 6px;'>Deterministic Rollout</div>
          <img src='{_data_uri(animation_path)}' style='width: 100%; border: 1px solid #ddd; border-radius: 8px;' />
        </div>
      </div>
    </div>
    """
))

triptych_path, uncertainty_path, animation_path


## Baseline protocol

- `deterministic_eval(stochastic=False)` использовать как воспроизводимый benchmark.
- `stochastic_eval(stochastic=True, K rollouts)` использовать как вероятностную оценку динамики.
- Все публичные heatmap и основные метрики считаются только по `primary_channel`.
- По умолчанию сохраняется старый режим: `data_channels=1`, `hidden_channels=1`, `primary_channel=0`, `loss_channels='primary'`.
